# Spain: counts, rates and observed crash distributions

Real published results from DGT, INE and Eurostat. Author: Javier Saguar.

See [methodology](../docs/methodology.md), [sources](../docs/sources.md) and [limitations](../docs/limitations.md).

Reusable implementations live in `src/`; reproduce artifacts using the CLI before rerunning these explorations.

In [1]:
from pathlib import Path
import json
import pandas as pd
from ebdi.metrics.index import composite
from ebdi.utils.io import read_yaml
ROOT = Path.cwd() if (Path.cwd() / "configs").exists() else Path.cwd().parent
TABLES = ROOT / "outputs/tables"
spain = pd.read_csv(TABLES / "spain_metrics.csv", dtype={"province_code": str})


## National reconciliation

In [2]:
national = pd.read_csv(TABLES / "national_totals.csv")
assert national.injury_crashes.sum() == 301218
print(national.to_string(index=False))

 year  injury_crashes  fatalities  hospitalized  population  urban_crashes  rear_lateral_crashes  severe_crashes  unknown_collision  injury_crashes_per_100k_population
 2022           97916        1746          8502    47486727          64616                 45605            9030                  0                          206.196565
 2023          101306        1806          9265    48085361          65976                 47830            9801                  0                          210.679504
 2024          101996        1785          9561    48619695          66224                 48306           10019                 35                          209.783299


## Does changing the measure change the territorial ranking?

In [3]:
latest = spain.loc[spain.year.eq(2024)]
columns = ["injury_crashes", "injury_crashes_per_100k_population", "fatalities_per_100k_population"]
print(latest[columns].corr(method="spearman").round(3).to_string())
for column in columns[1:]:
    print(column)
    print(latest.nlargest(5, column)[["province", "injury_crashes", "fatalities", column]].to_string(index=False))

                                    injury_crashes  injury_crashes_per_100k_population  fatalities_per_100k_population
injury_crashes                               1.000                               0.455                          -0.550
injury_crashes_per_100k_population           0.455                               1.000                          -0.352
fatalities_per_100k_population              -0.550                              -0.352                           1.000
injury_crashes_per_100k_population
 province  injury_crashes  fatalities  injury_crashes_per_100k_population
  Melilla             413           1                          480.316334
    Ceuta             300           1                          360.667957
   Lleida            1542          32                          341.371730
Barcelona           17945         130                          305.307952
   Málaga            5185          65                          292.161891
fatalities_per_100k_population
province  inju

## When and under what recorded circumstances? Counts are not travel risk

In [4]:
groups = pd.read_csv(TABLES / "crash_breakdowns.csv")
counts = groups.groupby(["year", "dimension"]).injury_crashes.sum().unstack()
assert counts.eq(national.set_index("year").injury_crashes, axis=0).all().all()
print(groups.loc[groups.year.eq(2024) & groups.dimension.eq("TIPO_VIA"), ["label", "injury_crashes", "severe_crashes", "conditional_severe_fraction", "small_sample"]].to_string(index=False))

                                  label  injury_crashes  severe_crashes  conditional_severe_fraction  small_sample
                     Autopista de peaje             425              51                     0.120000         False
                        Autopista libre            3316             291                     0.087756         False
                                Autovía            8136             831                     0.102139         False
                   Vía para automóviles              74              12                     0.162162         False
Carretera Convencional de doble calzada            1769             172                     0.097230         False
Carretera Convencional de calzada única           20281            3135                     0.154578         False
                        Vía de servicio             384              50                     0.130208         False
                        Ramal de enlace             322              16         